# Mid-circuit measurement with DSP classification

This notebook runs a single-qubit mid-circuit measurement experiment with `Experiment.execute(...)`. Each repetition is a single-shot experiment (`mode="single"`), and we repeat it many times to confirm that the first and second readout outcomes cluster on `gg` and `ee`.

The DSP classifier is enabled with `classification_source="gmm_linear"`, which derives the hardware decision line from the stored GMM state centers.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import qubex as qx


In [ ]:
exp = qx.Experiment(
    chip_id="",
    muxes=[],
)
exp.connect()

target = exp.qubit_labels[2]
readout_target = exp.ctx.resolve_read_label(target)
target, readout_target


In [ ]:
#exp.configure()


In [ ]:
exp.tool.print_target_frequencies(exp.qubits)

In [ ]:
# Prepare the X90 pulse and store the GMM state centers used by gmm_linear.
exp.obtain_rabi_params([target], plot=True)
exp.calibrate_hpi_pulse([target], plot=True)
exp.build_classifier([target], n_states=2, n_shots=10000, plot=True)


In [ ]:
wait_ns = 512
readout_duration = int(round(exp.readout_duration))
readout_pre_margin = int(round(exp.readout_pre_margin))
readout_post_margin = int(round(exp.readout_post_margin))

readout_pulse = exp.readout(
    readout_target,
    duration=readout_duration,
    pre_margin=readout_pre_margin,
    post_margin=readout_post_margin,
)

with qx.PulseSchedule([target, readout_target]) as sequence:
    sequence.add(target, exp.hpi_pulse[target])
    sequence.barrier()
    sequence.add(readout_target, readout_pulse)
    sequence.barrier()
    sequence.add(target, qx.Blank(wait_ns))
    sequence.add(readout_target, qx.Blank(wait_ns))
    sequence.barrier()
    sequence.add(readout_target, readout_pulse)

sequence


In [ ]:
sequence.plot()


In [ ]:
# Each repetition is a single-shot DSP-classified experiment.
n_analysis_shots = 10000
shot_interval = 150 * 1024

result = exp.execute(
    schedule=sequence,
    mode="single",
    n_shots=n_analysis_shots,
    shot_interval=shot_interval,
    enable_dsp_sum=True,
    enable_dsp_demodulation=True,
    enable_dsp_classification=True,
    classification_source="gmm_linear",
    plot=False,
)

len(result.data[target])


In [ ]:
first_capture = result.data[target][0]
second_capture = result.data[target][1]

print("first raw DSP values:", np.unique(first_capture.raw))
print("second raw DSP values:", np.unique(second_capture.raw))
print("first logical labels:", np.unique(first_capture.classified))
print("second logical labels:", np.unique(second_capture.classified))

assert set(np.unique(first_capture.raw)).issubset({0, 3})
assert set(np.unique(second_capture.raw)).issubset({0, 3})

# 0 -> g, 1 -> e
shots = result.get_classified_data(targets=[(target, 0), (target, 1)])
shots[:10]


In [ ]:
bit_counts = result.get_counts(targets=[(target, 0), (target, 1)])
ordered_labels = ["00", "01", "10", "11"]
counts = np.array([bit_counts.get(label, 0) for label in ordered_labels])
probabilities = counts / counts.sum()

for label, count, prob in zip(ordered_labels, counts, probabilities, strict=True):
    print(f"P({label}) = {prob:.4f} ({count} shots)")

diagonal_probability = probabilities[0] + probabilities[3]
print(f"P(gg) + P(ee) = {diagonal_probability:.4f}")


In [ ]:
probability_matrix = np.array(
    [
        [bit_counts.get("00", 0), bit_counts.get("01", 0)],
        [bit_counts.get("10", 0), bit_counts.get("11", 0)],
    ],
    dtype=float,
)
probability_matrix /= probability_matrix.sum()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(ordered_labels, counts, color=["#4c78a8", "#f58518", "#e45756", "#54a24b"])
axes[0].set_title(f"{target}: classified bitstrings")
axes[0].set_xlabel("bitstring (0=g, 1=e)")
axes[0].set_ylabel("shots")

im = axes[1].imshow(probability_matrix, vmin=0.0, vmax=probability_matrix.max())
axes[1].set_title("First vs second measurement")
axes[1].set_xticks([0, 1], labels=["2nd: g", "2nd: e"])
axes[1].set_yticks([0, 1], labels=["1st: g", "1st: e"])

for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{probability_matrix[i, j]:.3f}", ha="center", va="center", color="white")

fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="probability")
fig.tight_layout()


If the calibration and waiting time are appropriate, the dominant populations should remain on the diagonal (`gg` and `ee`). Off-diagonal weight (`ge` or `eg`) indicates assignment error, relaxation during the wait, or insufficient classifier calibration.
